# Zero-shot baselines (dev)

Two parts: generation runs on Kaggle (GPU T4 x2, Internet On, secret `HF_TOKEN` attached), scoring runs locally with the project venv.

## Generation (Kaggle)

Skip this part when you only score.

### Setup

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# prints the Hugging Face user of the token: accept the Gemma license with THIS account
from huggingface_hub import whoami
print(whoami(token=os.environ["HF_TOKEN"])["name"])

In [ ]:
!git clone https://github.com/mohinska/mmculture-qa.git
%cd mmculture-qa
!pip install -q transformers==4.57.6 accelerate pillow bitsandbytes qwen-omni-utils torchvision python-dotenv tqdm huggingface_hub
%env MMCQA_OUT=/kaggle/working

In [ ]:
!python shared/download_data.py dev --task qa

## Qwen2.5-Omni-3B (official reference baseline)

Full dev (1000 items), 4-bit, prompt `official`. Resumable: rerun the cell after a crash.

In [ ]:
# !python t2_text/run_vlm.py --model Qwen/Qwen2.5-Omni-3B --prompt-id official --n 0 --load-4bit --dtype float16

## Gemma 3 4B

Accept the license at huggingface.co/google/gemma-3-4b-it with the account printed above, otherwise the download fails with a 401 error. Run it with bfloat16; `--load-4bit` is the fallback.

In [ ]:
# !python t2_text/run_vlm.py --model google/gemma-3-4b-it --prompt-id plain --n 200 --dtype bfloat16

In [ ]:
# !python t2_text/run_vlm.py --model google/gemma-3-4b-it --prompt-id gold_style_v1 --n 200 --dtype bfloat16

## Qwen3-VL-4B and 8B

Answers already exist, so these cells are commented out.

In [ ]:
# !python t2_text/run_vlm.py --model Qwen/Qwen3-VL-4B-Instruct --prompt-id plain --n 200
# !python t2_text/run_vlm.py --model Qwen/Qwen3-VL-4B-Instruct --prompt-id gold_style_v1 --n 200

In [ ]:
# !python t2_text/run_vlm.py --model Qwen/Qwen3-VL-8B-Instruct --prompt-id plain --n 200
# !python t2_text/run_vlm.py --model Qwen/Qwen3-VL-8B-Instruct --prompt-id gold_style_v1 --n 200

## Qwen3-VL-4B, full dev (best so far)

`gold_style_v1`, all 1000 dev items. Resumable: the 200 items of the subset run are in the same preds file and are skipped, so only 800 run. After it finishes, download `preds/` and rescore locally (the Scoring cell rescores it because its n changed).

In [ ]:
!python t2_text/run_vlm.py --model Qwen/Qwen3-VL-4B-Instruct --prompt-id gold_style_v1 --n 0

## Download

Then: right panel > Data > Output > the zip > Download. Put the files into `preds/` locally.

In [ ]:
!cd /kaggle/working && zip -r preds_$(date +%m%d_%H%M).zip preds

## Scoring

Runs locally, not on Kaggle. Prerequisites:

- the project venv with `pip install -r requirements.txt` (Apple Silicon Mac or Linux)
- predictions in `preds/` (`t2_text_dev_en_*.jsonl`, `{id, answer, lang}` rows)

`evaluate()` from `shared/metrics.py` is the only scoring path. It writes `results/<name>.json` and `preds/<name>.scores.jsonl`. The first run downloads about 1 GB of mmBERT weights.

In [7]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")  # scoring runs locally; the cells below do nothing on Kaggle
if ON_KAGGLE:
    print("Skipping scoring on Kaggle: download preds/ and score locally.")

In [8]:
if not ON_KAGGLE:
    import json, subprocess, sys, tempfile
    from pathlib import Path

    REPO = Path.cwd().resolve().parents[1]  # notebook lives in t2_text/notebooks/
    sys.path.insert(0, str(REPO / "shared"))
    sys.path.insert(0, str(REPO / "t2_text"))
    from data import PREDS_DIR, RESULTS_DIR, load_split, load_jsonl
    from metrics import evaluate
    from prompts import PROMPTS

    import pandas as pd

    gold = {r["id"]: r["answer"] for r in load_split("qa", "dev")}

In [9]:
if not ON_KAGGLE:
    # preds files with at least one non-empty answer
    files = []
    for f in sorted(PREDS_DIR.glob("t2_text_dev_en_*.jsonl")):
        if f.name.endswith(".scores.jsonl"):
            continue
        if not any(str(r.get("answer") or "").strip() for r in load_jsonl(f)):
            print("skip (all answers empty):", f.name)
            continue
        files.append(f)
    print(len(files), "files to score")

7 files to score


In [10]:
if not ON_KAGGLE:
    FORCE = False  # True: rescore even when results/<name>.json already covers all rows
    for f in files:
        res = RESULTS_DIR / f"{f.stem}.json"
        if not FORCE and res.exists() and json.loads(res.read_text(encoding="utf-8"))["n"] == len(load_jsonl(f)):
            print("already scored:", f.name)
            continue
        print("scoring", f.name)
        try:
            evaluate(f, "dev")
        except Exception as e:
            print("FAILED:", f.name, type(e).__name__, e)

scoring t2_text_dev_en_Qwen2.5-Omni-3B_official.jsonl
[mmcqa-eval] phase=dev language=en items=1000 file=prediction.jsonl
[mmcqa-eval] RESULT dev/en: F1=0.7396 BLEU=6.91 ROUGE-L=0.2506 coverage=1.0 (38.1s)
already scored: t2_text_dev_en_Qwen3-VL-4B-Instruct_gold_style_v1.jsonl
already scored: t2_text_dev_en_Qwen3-VL-4B-Instruct_plain.jsonl
already scored: t2_text_dev_en_Qwen3-VL-8B-Instruct_gold_style_v1.jsonl
already scored: t2_text_dev_en_Qwen3-VL-8B-Instruct_plain.jsonl
scoring t2_text_dev_en_gemma-3-4b-it_gold_style_v1.jsonl
[mmcqa-eval] phase=dev language=en items=200 file=prediction.jsonl
[mmcqa-eval] RESULT dev/en: F1=0.7690 BLEU=7.35 ROUGE-L=0.2733 coverage=1.0 (6.6s)
scoring t2_text_dev_en_gemma-3-4b-it_plain.jsonl
[mmcqa-eval] phase=dev language=en items=200 file=prediction.jsonl
[mmcqa-eval] RESULT dev/en: F1=0.7070 BLEU=3.22 ROUGE-L=0.2038 coverage=1.0 (10.3s)


In [11]:
if not ON_KAGGLE:
    def split_stem(stem):
        """t2_text_dev_en_<model>_<prompt> -> (model, prompt); prompt is a key of PROMPTS."""
        tail = stem.removeprefix("t2_text_dev_en_")
        for prompt in PROMPTS:
            if tail.endswith("_" + prompt):
                return tail[:-len(prompt) - 1], prompt
        return tail, "?"

    table = []
    for f in files:
        out = RESULTS_DIR / f"{f.stem}.json"
        if not out.exists():
            continue
        r = json.loads(out.read_text(encoding="utf-8"))
        s = r["submission_result"]
        model, prompt = split_stem(f.stem)
        table.append({"model": model, "prompt": prompt, "n": r["n"],
                      "BERTScore-F1": s["BERTScore-F1"], "P": s["BERTScore-P"], "R": s["BERTScore-R"],
                      "BLEU": s["BLEU"], "ROUGE-L": s["ROUGE-L"], "coverage": s["Coverage"]})
    display(pd.DataFrame(table).sort_values("BERTScore-F1", ascending=False).reset_index(drop=True))

,model,prompt,n,BERTScore-F1,P,R,BLEU,ROUGE-L,coverage
0,Qwen3-VL-4B-Instruct,gold_style_v1,200,0.791486,0.785170,0.800175,11.5740,0.323889,1.0
1,Qwen3-VL-8B-Instruct,gold_style_v1,200,0.787984,0.777307,0.801108,11.4800,0.328921,1.0
2,gemma-3-4b-it,gold_style_v1,200,0.769030,0.742979,0.799031,7.3483,0.273259,1.0
3,Qwen2.5-Omni-3B,official,1000,0.739577,0.691612,0.797276,6.9132,0.250641,1.0
4,Qwen3-VL-4B-Instruct,plain,200,0.732188,0.664444,0.817103,3.7009,0.232337,1.0
5,Qwen3-VL-8B-Instruct,plain,200,0.725242,0.661588,0.804494,3.5450,0.210505,1.0
6,gemma-3-4b-it,plain,200,0.707011,0.641382,0.789381,3.2191,0.203758,1.0


In [12]:
if not ON_KAGGLE:
    def med_words(texts):
        return pd.Series([len(str(t).split()) for t in texts]).median()

    for f in files:
        rows = load_jsonl(f)
        print(f"{f.stem.removeprefix('t2_text_dev_en_'):45s} pred median {med_words(r['answer'] for r in rows):5.1f}"
              f"  gold median {med_words(gold[r['id']] for r in rows):5.1f}")

Qwen2.5-Omni-3B_official                      pred median  50.0  gold median  24.0
Qwen3-VL-4B-Instruct_gold_style_v1            pred median  24.0  gold median  25.0
Qwen3-VL-4B-Instruct_plain                    pred median  70.0  gold median  25.0
Qwen3-VL-8B-Instruct_gold_style_v1            pred median  24.0  gold median  25.0
Qwen3-VL-8B-Instruct_plain                    pred median  70.0  gold median  25.0
gemma-3-4b-it_gold_style_v1                   pred median  34.0  gold median  25.0
gemma-3-4b-it_plain                           pred median  66.0  gold median  25.0
